In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.preprocessing.image import img_to_array
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from imutils import paths
import matplotlib.pyplot as plt
import numpy as np
import os
from keras.callbacks import ModelCheckpoint

In [ ]:
DIRECTORY = 'New folder'
CATEGORIES = ['mask_weared_incorrect','with_mask','without_mask']

data = []
labels = []

for category in CATEGORIES:
    path = os.path.join(DIRECTORY, category)
    for img in os.listdir(path):
        img_path = os.path.join(path, img)
        image = load_img(img_path, target_size=(128, 128))
        image = img_to_array(image)
        image = preprocess_input(image)

        data.append(image)
        labels.append(category)

In [ ]:
classes = ['mask weared incorrect','with mask','without mask']

In [ ]:
labels = np.array(labels)
print(labels.shape)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
labels = le.fit_transform(labels)

data = np.array(data)/255.0

In [ ]:
labels

In [ ]:
(trainX, testX, trainY, testY) = train_test_split(data, labels,test_size=0.20, stratify=labels, random_state=42)

In [ ]:
trainY

In [ ]:
testY

In [ ]:
cnn = models.Sequential([
    layers.Conv2D(filters = 200 , kernel_size = (3,3) , activation = 'relu' , input_shape = (128, 128, 3)),
    layers.MaxPooling2D((2,2)),
    
    layers.Conv2D(filters = 100 , kernel_size = (3,3) , activation = 'relu'),
    layers.MaxPooling2D((2,2)),
    
    layers.Flatten(),
    layers.Dense(50, activation = 'relu'),
    layers.Dense(3, activation = 'softmax'),
])

In [ ]:
cnn.compile(optimizer = 'adam',
              loss = 'sparse_categorical_crossentropy',
              metrics = ['accuracy']
)

In [ ]:
checkpoint = ModelCheckpoint('model-{epoch:03d}.model',monitor='val_loss',verbose=0,save_best_only=True,mode='auto')
history=cnn.fit(trainX,trainY,epochs=20,callbacks=[checkpoint],validation_split=0.2)

In [ ]:
cnn.evaluate(testX , testY)

In [ ]:
y_pred = cnn.predict(testX)
y_pred_label = [np.argmax(i) for i in y_pred]
y_pred_label[0]

In [ ]:
classes[y_pred_label[0]]

In [ ]:
from matplotlib import pyplot as plt

plt.plot(history.history['loss'],'r',label='training loss')
plt.plot(history.history['val_loss'],label='validation loss')
plt.xlabel('# epochs')
plt.ylabel('loss')
plt.legend()
plt.show()

In [ ]:
plt.plot(history.history['accuracy'],'r',label='training accuracy')
plt.plot(history.history['val_accuracy'],label='validation accuracy')
plt.xlabel('# epochs')
plt.ylabel('accuracy')
plt.legend()
plt.show()